# 3×3 Rydberg quench 全后端 benchmark

这个 notebook 用同一套 3×3 `1r` 方格晶格和同一条失谐扫描 protocol，对当前仓库暴露的模拟路径做 benchmark。结构保持直写：每个算法占三个 block，第一个 block 说明算法，第二个 block 运行并计时，第三个 block 画该算法的逐格点 Rydberg occupation 热图。

没有 `RUN_*` 开关。要得到某个算法的结果，运行前面的参数/protocol/system block，再直接运行该算法自己的运行 block 和画图 block。所有 TN 后端都通过同一个 `system` lowering，`system` 使用 `InteractionSpec(C6=C6_70s, mode="nn")`，因此 TN interaction 也是最近邻设定。


In [ ]:
import importlib
import time

import numpy as np
import matplotlib.pyplot as plt
import ryd_gate as rg
import ryd_gate.protocols.sweep as rg_sweep
from ryd_gate import InteractionSpec
from ryd_gate.lattice import make_square_lattice
from ryd_gate.backends.tn_common.compiler import tn_lattice_spec_from_system

# Refresh SweepProtocol when rerunning this notebook in an already-started kernel.
rg_sweep = importlib.reload(rg_sweep)
rg.SweepProtocol = rg_sweep.SweepProtocol

# Experimental-style parameters
a_um = 6.8                         # lattice spacing, um
Omega = 2 * np.pi * 3.8e6          # rad/s
C6_70s = 2 * np.pi * 874e9         # rad/s * um^6, Rb 70S typical

delta_start = -2 * np.pi * 8.0e6   # rad/s
delta_end = 2 * np.pi * 8.0e6      # rad/s
t_sweep = 1.5e-6                   # s
omega_ramp_frac = 0.1
N_sites = 3 * 3


In [ ]:
# No local-addressing detuning shift for this run.
local_detuning_offsets = np.zeros(N_sites)

# Effectively global Rabi
def omega_half_t(t):
    ramp_time = omega_ramp_frac * t_sweep
    if ramp_time == 0:
        return 0.5 * Omega
    return 0.5 * Omega * min(1.0, max(0.0, t / ramp_time))

# 全局 Rydberg 失谐：Omega ramp 结束后才开始 chirp
def delta_t(t):
    ramp_time = omega_ramp_frac * t_sweep
    if t <= ramp_time:
        return delta_start
    chirp_time = max(t_sweep - ramp_time, np.finfo(float).eps)
    frac = np.clip((t - ramp_time) / chirp_time, 0.0, 1.0)
    return delta_start + (delta_end - delta_start) * frac

# 局部 addressing shift；这里全部为 0
def delta_local(t, ind):
    return local_detuning_offsets[ind]

# 1. 几何 + 能级结构：3x3 方格，每个格点是 |1>(基态) / |r>(Rydberg) 两能级
geom = make_square_lattice(3, 3, spacing_um=a_um)

# 2. 自定义 sweep：global Delta(t) 和 local address shift 分开输入
protocol = rg.SweepProtocol(
    t_gate=t_sweep,
    omega_half_fn=omega_half_t,
    delta_fn=delta_t,
    address_fn=delta_local,
    n_steps=120,
)
system = rg.RydbergSystem.from_lattice(
    geom, "1r",
    interaction=InteractionSpec(C6=C6_70s, mode="nn"),       # 近邻 VdW；TN 勿用默认 all
    protocol=protocol,
)
params = system.unpack_params([])
t_eval = np.linspace(0.0, t_sweep, 7)

protocol.plot(system=system, params=params, savefig=False)


In [ ]:
# Benchmark 公共量：只放数据容器和几何尺寸，不放算法开关。
dt_tn = 0.2 / Omega  # seconds; 0.2 * Omega^{-1}, same time unit as t_sweep/t_eval
coords = np.asarray(geom.coords)
x_vals = np.unique(coords[:, 0])
y_vals = np.unique(coords[:, 1])
Lx, Ly = len(x_vals), len(y_vals)

# 确认 TN lowering 继承 system 的最近邻 interaction pair list。
tn_spec = tn_lattice_spec_from_system(system)
pair_distances = np.asarray([
    np.linalg.norm(coords[int(i)] - coords[int(j)])
    for i, j, _ in tn_spec.vdw_pairs
])
print(f"TN pairs: {len(tn_spec.vdw_pairs)} bonds; nearest-neighbor only = {np.allclose(pair_distances, a_um)}")

benchmark_results = {}
benchmark_rows = []


## 3. Exact state-vector baseline

下面两个 block 使用 `backend="exact"`。exact 后端直接保存每个 `t_eval` 的态矢量，平均 Rydberg occupation 和逐格点 occupation 都用 `system.expectation(...)` 从态矢量读出。这个结果作为后面所有近似后端的数值基准。


In [ ]:
# 3a. 精确后端：states 是态矢量，用 system.expectation 读 Rydberg 均值和逐格点 occupation。
method = "exact"
_t0 = time.perf_counter()
res = rg.simulate(system, [], "all_ground", backend="exact", t_eval=t_eval)
exact_elapsed = time.perf_counter() - _t0

exact_n_mean = np.asarray([
    system.expectation("sum_nr", psi) / system.N
    for psi in res.states
])
exact_n_i = np.asarray([
    [system.expectation(f"n_r_{i}", psi) for i in range(system.N)]
    for psi in res.states
])

benchmark_results[method] = {
    "label": "Exact state-vector",
    "status": "ok",
    "elapsed_s": exact_elapsed,
    "times": np.asarray(res.times),
    "n_mean": exact_n_mean,
    "n_i": exact_n_i,
    "max_abs_diff_n_mean": 0.0,
    "max_abs_diff_n_i": 0.0,
}
benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
benchmark_rows.append({
    "method": method,
    "status": "ok",
    "elapsed_s": exact_elapsed,
    "max_abs_diff_n_mean": 0.0,
    "max_abs_diff_n_i": 0.0,
    "error": "",
})

print(f"Exact elapsed: {exact_elapsed:.3f} s")
print("time(us)  exact <n_r>")
for t, exact_val in zip(res.times * 1e6, exact_n_mean):
    print(f"{t:7.3f}   {exact_val:10.4f}")


In [ ]:
# 3b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if exact_n_i is None:
    print("exact has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res.times, exact_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("Exact per-site Rydberg occupation", y=1.05)
    plt.show()


## 4. TeNPy CPU MPS-TDVP

下面两个 block 使用 `backend="tenpy"`。它把 3×3 二维晶格按 snake ordering 拉平成一维 MPS，然后用 TeNPy TDVP 做时间演化。主要参数是 `chi_max` 和 `dt`；这里所有 interaction 来自前面的 `system`，也就是最近邻 `nn` pair list。


In [ ]:
# 4a. TeNPy MPS-TDVP：TN 后端把 <n_r> 和每个格点 <n_i> 记录在 metadata["obs"] 里。
method = "tenpy"
try:
    _t0 = time.perf_counter()
    res_tenpy = rg.simulate(
        system, [], "all_ground",
        backend="tenpy", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"chi_max": 32, "dt": dt_tn, "svd_min": 1e-10},
    )
    tenpy_elapsed = time.perf_counter() - _t0
    tenpy_n_mean = np.asarray(res_tenpy.metadata["obs"]["n_mean"])
    tenpy_n_i = np.asarray(res_tenpy.metadata["obs"]["n_i"])

    tenpy_diff_mean = float(np.max(np.abs(tenpy_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    tenpy_diff_i = float(np.max(np.abs(tenpy_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "TeNPy MPS-TDVP", "status": "ok", "elapsed_s": tenpy_elapsed, "times": np.asarray(res_tenpy.times), "n_mean": tenpy_n_mean, "n_i": tenpy_n_i, "max_abs_diff_n_mean": tenpy_diff_mean, "max_abs_diff_n_i": tenpy_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": tenpy_elapsed, "max_abs_diff_n_mean": tenpy_diff_mean, "max_abs_diff_n_i": tenpy_diff_i, "error": ""})

    print(f"TeNPy elapsed: {tenpy_elapsed:.3f} s")
    print("time(us)  exact <n_r>  TeNPy <n_r>")
    for t, exact_val, tn_val in zip(t_eval * 1e6, exact_n_mean, tenpy_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {tn_val:10.4f}")
    print(f"max |delta <n_r>| = {tenpy_diff_mean:.3e}; max |delta <n_i>| = {tenpy_diff_i:.3e}")
except Exception as exc:
    tenpy_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    tenpy_n_mean = None
    tenpy_n_i = None
    tenpy_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "TeNPy MPS-TDVP", "status": "failed", "elapsed_s": tenpy_elapsed, "error": tenpy_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": tenpy_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": tenpy_error})
    print(f"TeNPy failed after {tenpy_elapsed:.3f} s")
    print(tenpy_error)


In [ ]:
# 4b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if tenpy_n_i is None:
    print("tenpy has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_tenpy.times, tenpy_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("TeNPy MPS-TDVP per-site Rydberg occupation", y=1.05)
    plt.show()


## 5. YASTN MPS-TDVP (`mps_gpu`)

下面两个 block 使用 `backend="mps_gpu"`，也就是 YASTN 实现的 MPS-TDVP 路径。当前设置 `use_cuda=False`，CUDA 机器上可以把它改成 `True`。它和 TeNPy 都是一维 MPS 近似，差别主要来自实现和可用硬件。


In [ ]:
# 5a. GPU-capable MPS-TDVP 后端（YASTN；CUDA 机器上把 use_cuda=True）。
method = "mps_gpu"
try:
    _t0 = time.perf_counter()
    res_mps_gpu = rg.simulate(
        system, [], "all_ground",
        backend="mps_gpu", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"chi_max": 12, "dt": dt_tn, "svd_min": 1e-10, "use_cuda": False},
    )
    mps_gpu_elapsed = time.perf_counter() - _t0
    mps_gpu_n_mean = np.asarray(res_mps_gpu.metadata["obs"]["n_mean"])
    mps_gpu_n_i = np.asarray(res_mps_gpu.metadata["obs"]["n_i"])

    mps_gpu_diff_mean = float(np.max(np.abs(mps_gpu_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    mps_gpu_diff_i = float(np.max(np.abs(mps_gpu_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "YASTN MPS-TDVP", "status": "ok", "elapsed_s": mps_gpu_elapsed, "times": np.asarray(res_mps_gpu.times), "n_mean": mps_gpu_n_mean, "n_i": mps_gpu_n_i, "max_abs_diff_n_mean": mps_gpu_diff_mean, "max_abs_diff_n_i": mps_gpu_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": mps_gpu_elapsed, "max_abs_diff_n_mean": mps_gpu_diff_mean, "max_abs_diff_n_i": mps_gpu_diff_i, "error": ""})

    print(f"YASTN MPS elapsed: {mps_gpu_elapsed:.3f} s")
    print("time(us)  exact <n_r>  MPS-GPU <n_r>")
    for t, exact_val, mps_val in zip(t_eval * 1e6, exact_n_mean, mps_gpu_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {mps_val:9.4f}")
    print(f"max |delta <n_r>| = {mps_gpu_diff_mean:.3e}; max |delta <n_i>| = {mps_gpu_diff_i:.3e}")
except Exception as exc:
    mps_gpu_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    mps_gpu_n_mean = None
    mps_gpu_n_i = None
    mps_gpu_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "YASTN MPS-TDVP", "status": "failed", "elapsed_s": mps_gpu_elapsed, "error": mps_gpu_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": mps_gpu_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": mps_gpu_error})
    print(f"YASTN MPS failed after {mps_gpu_elapsed:.3f} s")
    print(mps_gpu_error)


In [ ]:
# 5b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if mps_gpu_n_i is None:
    print("mps_gpu has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_mps_gpu.times, mps_gpu_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("YASTN MPS-TDVP per-site Rydberg occupation", y=1.05)
    plt.show()


## 6. PyTreeNet CPU TTN-TDVP

下面两个 block 使用 `backend="ttn"`。这是 vendored PyTreeNet 的二叉树 TTN-TDVP，使用固定 `chi_max` 的 one-site TDVP。TTN 不把二维系统压成一条 MPS 链，但树结构和时变 Hamiltonian 的构造会带来额外开销。


In [ ]:
# 6a. PyTreeNet TTN-TDVP：CPU-only one-site TTN TDVP。
method = "ttn"
try:
    _t0 = time.perf_counter()
    res_ttn = rg.simulate(
        system, [], "all_ground",
        backend="ttn", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"chi_max": 8, "dt": dt_tn, "tdvp_order": 1, "initial_noise": 1e-10, "seed": 0},
    )
    ttn_elapsed = time.perf_counter() - _t0
    ttn_n_mean = np.asarray(res_ttn.metadata["obs"]["n_mean"])
    ttn_n_i = np.asarray(res_ttn.metadata["obs"]["n_i"])

    ttn_diff_mean = float(np.max(np.abs(ttn_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    ttn_diff_i = float(np.max(np.abs(ttn_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "PyTreeNet TTN-TDVP", "status": "ok", "elapsed_s": ttn_elapsed, "times": np.asarray(res_ttn.times), "n_mean": ttn_n_mean, "n_i": ttn_n_i, "max_abs_diff_n_mean": ttn_diff_mean, "max_abs_diff_n_i": ttn_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": ttn_elapsed, "max_abs_diff_n_mean": ttn_diff_mean, "max_abs_diff_n_i": ttn_diff_i, "error": ""})

    print(f"TTN elapsed: {ttn_elapsed:.3f} s")
    print("time(us)  exact <n_r>  TTN <n_r>")
    for t, exact_val, ttn_val in zip(t_eval * 1e6, exact_n_mean, ttn_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {ttn_val:10.4f}")
    print(f"max |delta <n_r>| = {ttn_diff_mean:.3e}; max |delta <n_i>| = {ttn_diff_i:.3e}")
except Exception as exc:
    ttn_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    ttn_n_mean = None
    ttn_n_i = None
    ttn_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "PyTreeNet TTN-TDVP", "status": "failed", "elapsed_s": ttn_elapsed, "error": ttn_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": ttn_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": ttn_error})
    print(f"TTN failed after {ttn_elapsed:.3f} s")
    print(ttn_error)


In [ ]:
# 6b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if ttn_n_i is None:
    print("ttn has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_ttn.times, ttn_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("PyTreeNet TTN-TDVP per-site Rydberg occupation", y=1.05)
    plt.show()


## 7. cuTensorNet / CuPy GPUTN

下面两个 block 使用 `backend="gputn"`。它需要 CuPy、cuQuantum 和可见 NVIDIA GPU。这个 block 保持和其它 TN 一样的 `system` 最近邻 interaction；如果本机缺 CUDA 依赖，会把失败原因写进 benchmark 表，不中断后续 block。


In [ ]:
# 7a. cuTensorNet/CuPy GPUTN：真正 CUDA tensor-network kernel，可用时计入同一 benchmark。
method = "gputn"
try:
    _t0 = time.perf_counter()
    res_gputn = rg.simulate(
        system, [], "all_ground",
        backend="gputn", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"chi_max": 32, "dt": dt_tn, "svd_min": 1e-10, "require_gpu": True, "kernel": "auto"},
    )
    gputn_elapsed = time.perf_counter() - _t0
    gputn_n_mean = np.asarray(res_gputn.metadata["obs"]["n_mean"])
    gputn_n_i = np.asarray(res_gputn.metadata["obs"]["n_i"])

    gputn_diff_mean = float(np.max(np.abs(gputn_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    gputn_diff_i = float(np.max(np.abs(gputn_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "cuTensorNet GPUTN", "status": "ok", "elapsed_s": gputn_elapsed, "times": np.asarray(res_gputn.times), "n_mean": gputn_n_mean, "n_i": gputn_n_i, "max_abs_diff_n_mean": gputn_diff_mean, "max_abs_diff_n_i": gputn_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": gputn_elapsed, "max_abs_diff_n_mean": gputn_diff_mean, "max_abs_diff_n_i": gputn_diff_i, "error": ""})

    print(f"GPUTN elapsed: {gputn_elapsed:.3f} s")
    print("time(us)  exact <n_r>  GPUTN <n_r>")
    for t, exact_val, gpu_val in zip(t_eval * 1e6, exact_n_mean, gputn_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {gpu_val:10.4f}")
    print(f"max |delta <n_r>| = {gputn_diff_mean:.3e}; max |delta <n_i>| = {gputn_diff_i:.3e}")
except Exception as exc:
    gputn_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    gputn_n_mean = None
    gputn_n_i = None
    gputn_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "cuTensorNet GPUTN", "status": "failed", "elapsed_s": gputn_elapsed, "error": gputn_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": gputn_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": gputn_error})
    print(f"GPUTN failed after {gputn_elapsed:.3f} s")
    print(gputn_error)


In [ ]:
# 7b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if gputn_n_i is None:
    print("gputn has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_gputn.times, gputn_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("cuTensorNet GPUTN per-site Rydberg occupation", y=1.05)
    plt.show()


## 8. Julia ITensors MPS-TEBD

下面两个 block 使用 `backend="itensors"`。这是 Julia ITensors/ITensorMPS 的 MPS-TEBD bridge，仍然把二维格点按 snake ordering 放进 MPS。该 Julia kernel 记录 `sigma_z`，这里把它换算成 Rydberg occupation：`n_i = 0.5 * (sigma_z + 1)`。


In [ ]:
# 8a. Julia ITensors MPS-TEBD：从 metadata["obs"]["sigma_z"] 换算 <n_i>。
method = "itensors"
try:
    _t0 = time.perf_counter()
    res_itensors = rg.simulate(
        system, [], "all_ground",
        backend="itensors", t_eval=t_eval, observables=["sigma_z"],
        backend_options={"chi_max": 32, "dt": dt_tn, "svd_min": 1e-10, "timeout": 600},
    )
    itensors_elapsed = time.perf_counter() - _t0
    itensors_sigma_z = np.asarray(res_itensors.metadata["obs"]["sigma_z"])
    itensors_n_i = 0.5 * (itensors_sigma_z + 1.0)
    itensors_n_mean = itensors_n_i.mean(axis=1)

    itensors_diff_mean = float(np.max(np.abs(itensors_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    itensors_diff_i = float(np.max(np.abs(itensors_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "Julia ITensors MPS-TEBD", "status": "ok", "elapsed_s": itensors_elapsed, "times": np.asarray(res_itensors.times), "n_mean": itensors_n_mean, "n_i": itensors_n_i, "max_abs_diff_n_mean": itensors_diff_mean, "max_abs_diff_n_i": itensors_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": itensors_elapsed, "max_abs_diff_n_mean": itensors_diff_mean, "max_abs_diff_n_i": itensors_diff_i, "error": ""})

    print(f"ITensors elapsed: {itensors_elapsed:.3f} s")
    print("time(us)  exact <n_r>  ITensors <n_r>")
    for t, exact_val, it_val in zip(t_eval * 1e6, exact_n_mean, itensors_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {it_val:12.4f}")
    print(f"max |delta <n_r>| = {itensors_diff_mean:.3e}; max |delta <n_i>| = {itensors_diff_i:.3e}")
except Exception as exc:
    itensors_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    itensors_n_mean = None
    itensors_n_i = None
    itensors_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "Julia ITensors MPS-TEBD", "status": "failed", "elapsed_s": itensors_elapsed, "error": itensors_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": itensors_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": itensors_error})
    print(f"ITensors failed after {itensors_elapsed:.3f} s")
    print(itensors_error)


In [ ]:
# 8b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if itensors_n_i is None:
    print("itensors has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_itensors.times, itensors_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("Julia ITensors MPS-TEBD per-site Rydberg occupation", y=1.05)
    plt.show()


## 9. Julia ITensorNetworks GPU TTN-TDVP (`gputtn`)

下面两个 block 使用 `backend="gputtn"`。这是 Julia ITensorNetworks 的 GPU TTN-TDVP bridge，需要 Julia 环境和 CUDA 相关栈。它的结果同样进入计时和误差表；不可用时打印失败原因。


In [ ]:
# 9a. Julia ITensorNetworks GPU TTN-TDVP。
method = "gputtn"
try:
    _t0 = time.perf_counter()
    res_gputtn = rg.simulate(
        system, [], "all_ground",
        backend="gputtn", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"chi_max": 32, "dt": dt_tn, "svd_min": 1e-10, "timeout": 600, "use_cuda": True},
    )
    gputtn_elapsed = time.perf_counter() - _t0
    gputtn_n_mean = np.asarray(res_gputtn.metadata["obs"]["n_mean"])
    gputtn_n_i = np.asarray(res_gputtn.metadata["obs"]["n_i"])

    gputtn_diff_mean = float(np.max(np.abs(gputtn_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    gputtn_diff_i = float(np.max(np.abs(gputtn_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "ITensorNetworks GPU TTN-TDVP", "status": "ok", "elapsed_s": gputtn_elapsed, "times": np.asarray(res_gputtn.times), "n_mean": gputtn_n_mean, "n_i": gputtn_n_i, "max_abs_diff_n_mean": gputtn_diff_mean, "max_abs_diff_n_i": gputtn_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": gputtn_elapsed, "max_abs_diff_n_mean": gputtn_diff_mean, "max_abs_diff_n_i": gputtn_diff_i, "error": ""})

    print(f"GPUTTN elapsed: {gputtn_elapsed:.3f} s")
    print("time(us)  exact <n_r>  GPUTTN <n_r>")
    for t, exact_val, gpu_val in zip(t_eval * 1e6, exact_n_mean, gputtn_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {gpu_val:10.4f}")
    print(f"max |delta <n_r>| = {gputtn_diff_mean:.3e}; max |delta <n_i>| = {gputtn_diff_i:.3e}")
except Exception as exc:
    gputtn_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    gputtn_n_mean = None
    gputtn_n_i = None
    gputtn_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "ITensorNetworks GPU TTN-TDVP", "status": "failed", "elapsed_s": gputtn_elapsed, "error": gputtn_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": gputtn_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": gputtn_error})
    print(f"GPUTTN failed after {gputtn_elapsed:.3f} s")
    print(gputtn_error)


In [ ]:
# 9b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if gputtn_n_i is None:
    print("gputtn has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_gputtn.times, gputtn_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("ITensorNetworks GPU TTN-TDVP per-site Rydberg occupation", y=1.05)
    plt.show()


## 10. Julia TNQS 2D-TN / BP default backend

下面两个 block 使用 `backend="2dtn"` 且不指定 `engine_package`，因此走默认 Julia TensorNetworkQuantumSimulator/TNQS 2D-TN bridge。它保留二维图结构，测量输出 `sigma_z`，这里换算为 `n_i`。


In [ ]:
# 10a. Julia TNQS 2D-TN/BP：从 sigma_z 换算 Rydberg occupation。
method = "2dtn_tnqs"
try:
    _t0 = time.perf_counter()
    res_2dtn_tnqs = rg.simulate(
        system, [], "all_ground",
        backend="2dtn", t_eval=t_eval, observables=["sigma_z"],
        backend_options={"chi_max": 16, "dt": dt_tn, "svd_min": 1e-10, "measurement_alg": "bp", "measurement_bond_dim": 16, "timeout": 600},
    )
    tnqs_elapsed = time.perf_counter() - _t0
    tnqs_sigma_z = np.asarray(res_2dtn_tnqs.metadata["obs"]["sigma_z"])
    tnqs_n_i = 0.5 * (tnqs_sigma_z + 1.0)
    tnqs_n_mean = tnqs_n_i.mean(axis=1)

    tnqs_diff_mean = float(np.max(np.abs(tnqs_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    tnqs_diff_i = float(np.max(np.abs(tnqs_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "Julia TNQS 2D-TN/BP", "status": "ok", "elapsed_s": tnqs_elapsed, "times": np.asarray(res_2dtn_tnqs.times), "n_mean": tnqs_n_mean, "n_i": tnqs_n_i, "max_abs_diff_n_mean": tnqs_diff_mean, "max_abs_diff_n_i": tnqs_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": tnqs_elapsed, "max_abs_diff_n_mean": tnqs_diff_mean, "max_abs_diff_n_i": tnqs_diff_i, "error": ""})

    print(f"TNQS 2D-TN elapsed: {tnqs_elapsed:.3f} s")
    print("time(us)  exact <n_r>  TNQS <n_r>")
    for t, exact_val, tn_val in zip(t_eval * 1e6, exact_n_mean, tnqs_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {tn_val:10.4f}")
    print(f"max |delta <n_r>| = {tnqs_diff_mean:.3e}; max |delta <n_i>| = {tnqs_diff_i:.3e}")
except Exception as exc:
    tnqs_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    tnqs_n_mean = None
    tnqs_n_i = None
    tnqs_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "Julia TNQS 2D-TN/BP", "status": "failed", "elapsed_s": tnqs_elapsed, "error": tnqs_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": tnqs_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": tnqs_error})
    print(f"TNQS 2D-TN failed after {tnqs_elapsed:.3f} s")
    print(tnqs_error)


In [ ]:
# 10b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if tnqs_n_i is None:
    print("2dtn_tnqs has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_2dtn_tnqs.times, tnqs_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("Julia TNQS 2D-TN/BP per-site Rydberg occupation", y=1.05)
    plt.show()


## 11. YASTN finite-PEPS 2D-TN

下面两个 block 使用 `backend="2dtn"` 并设置 `engine_package="yastn"`。这是 Python YASTN finite-PEPS 路径，保留二维几何，当前用 CPU NumPy backend；CUDA 机器可以改成 YASTN 的 torch/cuda 设置。


In [ ]:
# 11a. YASTN finite-PEPS 2D-TN。
method = "2dtn_yastn"
try:
    _t0 = time.perf_counter()
    res_2dtn_yastn = rg.simulate(
        system, [], "all_ground",
        backend="2dtn", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"engine_package": "yastn", "chi_max": 8, "dt": dt_tn, "svd_min": 1e-10, "measurement_environment": "bp", "use_cuda": False},
    )
    yastn_2d_elapsed = time.perf_counter() - _t0
    yastn_2d_n_mean = np.asarray(res_2dtn_yastn.metadata["obs"]["n_mean"])
    yastn_2d_n_i = np.asarray(res_2dtn_yastn.metadata["obs"]["n_i"])

    yastn_2d_diff_mean = float(np.max(np.abs(yastn_2d_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    yastn_2d_diff_i = float(np.max(np.abs(yastn_2d_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "YASTN finite-PEPS 2D-TN", "status": "ok", "elapsed_s": yastn_2d_elapsed, "times": np.asarray(res_2dtn_yastn.times), "n_mean": yastn_2d_n_mean, "n_i": yastn_2d_n_i, "max_abs_diff_n_mean": yastn_2d_diff_mean, "max_abs_diff_n_i": yastn_2d_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": yastn_2d_elapsed, "max_abs_diff_n_mean": yastn_2d_diff_mean, "max_abs_diff_n_i": yastn_2d_diff_i, "error": ""})

    print(f"YASTN 2D-TN elapsed: {yastn_2d_elapsed:.3f} s")
    print("time(us)  exact <n_r>  YASTN-2D <n_r>")
    for t, exact_val, y_val in zip(t_eval * 1e6, exact_n_mean, yastn_2d_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {y_val:10.4f}")
    print(f"max |delta <n_r>| = {yastn_2d_diff_mean:.3e}; max |delta <n_i>| = {yastn_2d_diff_i:.3e}")
except Exception as exc:
    yastn_2d_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    yastn_2d_n_mean = None
    yastn_2d_n_i = None
    yastn_2d_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "YASTN finite-PEPS 2D-TN", "status": "failed", "elapsed_s": yastn_2d_elapsed, "error": yastn_2d_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": yastn_2d_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": yastn_2d_error})
    print(f"YASTN 2D-TN failed after {yastn_2d_elapsed:.3f} s")
    print(yastn_2d_error)


In [ ]:
# 11b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if yastn_2d_n_i is None:
    print("2dtn_yastn has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_2dtn_yastn.times, yastn_2d_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("YASTN finite-PEPS 2D-TN per-site Rydberg occupation", y=1.05)
    plt.show()


## 12. quimb PEPS 2D-TN

下面两个 block 使用 `backend="2dtn"` 并设置 `engine_package="quimb"`。它调用 quimb finite-PEPS / SimpleUpdate 路径，同样保留二维几何。CPU 默认用 NumPy；如果安装 CuPy，可把 `use_cuda=True` 和 `array_backend="cupy"`。


In [ ]:
# 12a. quimb PEPS 2D-TN。
method = "2dtn_quimb"
try:
    _t0 = time.perf_counter()
    res_2dtn_quimb = rg.simulate(
        system, [], "all_ground",
        backend="2dtn", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"engine_package": "quimb", "chi_max": 8, "dt": dt_tn, "svd_min": 1e-10, "algorithm": "simple_update", "use_cuda": False, "array_backend": "numpy"},
    )
    quimb_2d_elapsed = time.perf_counter() - _t0
    quimb_2d_n_mean = np.asarray(res_2dtn_quimb.metadata["obs"]["n_mean"])
    quimb_2d_n_i = np.asarray(res_2dtn_quimb.metadata["obs"]["n_i"])

    quimb_2d_diff_mean = float(np.max(np.abs(quimb_2d_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    quimb_2d_diff_i = float(np.max(np.abs(quimb_2d_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "quimb PEPS 2D-TN", "status": "ok", "elapsed_s": quimb_2d_elapsed, "times": np.asarray(res_2dtn_quimb.times), "n_mean": quimb_2d_n_mean, "n_i": quimb_2d_n_i, "max_abs_diff_n_mean": quimb_2d_diff_mean, "max_abs_diff_n_i": quimb_2d_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": quimb_2d_elapsed, "max_abs_diff_n_mean": quimb_2d_diff_mean, "max_abs_diff_n_i": quimb_2d_diff_i, "error": ""})

    print(f"quimb 2D-TN elapsed: {quimb_2d_elapsed:.3f} s")
    print("time(us)  exact <n_r>  quimb-2D <n_r>")
    for t, exact_val, q_val in zip(t_eval * 1e6, exact_n_mean, quimb_2d_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {q_val:10.4f}")
    print(f"max |delta <n_r>| = {quimb_2d_diff_mean:.3e}; max |delta <n_i>| = {quimb_2d_diff_i:.3e}")
except Exception as exc:
    quimb_2d_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    quimb_2d_n_mean = None
    quimb_2d_n_i = None
    quimb_2d_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "quimb PEPS 2D-TN", "status": "failed", "elapsed_s": quimb_2d_elapsed, "error": quimb_2d_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": quimb_2d_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": quimb_2d_error})
    print(f"quimb 2D-TN failed after {quimb_2d_elapsed:.3f} s")
    print(quimb_2d_error)


In [ ]:
# 12b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if quimb_2d_n_i is None:
    print("2dtn_quimb has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_2dtn_quimb.times, quimb_2d_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("quimb PEPS 2D-TN per-site Rydberg occupation", y=1.05)
    plt.show()


## 13. NetKet NQS-tVMC

下面两个 block 使用 `backend="nqs"` 并设置 `engine_package="netket"`。它用 neural quantum state 变分态和 real-time tVMC 演化。观测量来自 Monte Carlo 采样，因此和 exact 的差别包含变分误差、时间步长误差和采样噪声。


In [ ]:
# 13a. NetKet NQS-tVMC。
method = "nqs_netket"
try:
    _t0 = time.perf_counter()
    res_nqs_netket = rg.simulate(
        system, [], "all_ground",
        backend="nqs", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"engine_package": "netket", "dt": dt_tn, "n_samples": 1024, "n_chains": 16, "ansatz": "rbm", "alpha": 1.0, "seed": 1234, "sampler_seed": 5678, "use_cuda": False},
    )
    nqs_elapsed = time.perf_counter() - _t0
    nqs_n_mean = np.asarray(res_nqs_netket.metadata["obs"]["n_mean"])
    nqs_n_i = np.asarray(res_nqs_netket.metadata["obs"]["n_i"])

    nqs_diff_mean = float(np.max(np.abs(nqs_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    nqs_diff_i = float(np.max(np.abs(nqs_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "NetKet NQS-tVMC", "status": "ok", "elapsed_s": nqs_elapsed, "times": np.asarray(res_nqs_netket.times), "n_mean": nqs_n_mean, "n_i": nqs_n_i, "max_abs_diff_n_mean": nqs_diff_mean, "max_abs_diff_n_i": nqs_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": nqs_elapsed, "max_abs_diff_n_mean": nqs_diff_mean, "max_abs_diff_n_i": nqs_diff_i, "error": ""})

    print(f"NetKet NQS elapsed: {nqs_elapsed:.3f} s")
    print("time(us)  exact <n_r>  NQS <n_r>")
    for t, exact_val, nqs_val in zip(t_eval * 1e6, exact_n_mean, nqs_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {nqs_val:10.4f}")
    print(f"max |delta <n_r>| = {nqs_diff_mean:.3e}; max |delta <n_i>| = {nqs_diff_i:.3e}")
except Exception as exc:
    nqs_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    nqs_n_mean = None
    nqs_n_i = None
    nqs_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "NetKet NQS-tVMC", "status": "failed", "elapsed_s": nqs_elapsed, "error": nqs_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": nqs_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": nqs_error})
    print(f"NetKet NQS failed after {nqs_elapsed:.3f} s")
    print(nqs_error)


In [ ]:
# 13b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if nqs_n_i is None:
    print("nqs_netket has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_nqs_netket.times, nqs_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("NetKet NQS-tVMC per-site Rydberg occupation", y=1.05)
    plt.show()


## 14. 汇总对比图

这个 block 汇总已经运行过的算法，列出状态、墙钟时间、相对 exact 的最大平均 occupation 误差和最大逐格点 occupation 误差，并画运行时间、平均 Rydberg occupation 曲线和误差柱状图。失败后端会保留在文字表格中。


In [ ]:
print("full benchmark table:")
print(f"{'method':<14} {'status':<8} {'time(s)':>10} {'max|dn_mean|':>14} {'max|dn_i|':>12}")
for row in benchmark_rows:
    elapsed = row.get("elapsed_s", np.nan)
    err_mean = row.get("max_abs_diff_n_mean", np.nan)
    err_i = row.get("max_abs_diff_n_i", np.nan)
    elapsed_s = f"{elapsed:.3f}" if np.isfinite(elapsed) else "nan"
    err_mean_s = f"{err_mean:.3e}" if np.isfinite(err_mean) else "nan"
    err_i_s = f"{err_i:.3e}" if np.isfinite(err_i) else "nan"
    print(f"{row['method']:<14} {row['status']:<8} {elapsed_s:>10} {err_mean_s:>14} {err_i_s:>12}")
    if row.get("error"):
        print("  error:", row["error"])

ok_methods = [method for method, data in benchmark_results.items() if data.get("status") == "ok"]
if not ok_methods:
    print("No successful backend results to summarize.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

    elapsed = np.asarray([benchmark_results[m]["elapsed_s"] for m in ok_methods], dtype=float)
    axes[0].bar(ok_methods, elapsed, color="tab:blue", alpha=0.8)
    axes[0].set_ylabel("wall time (s)")
    axes[0].set_title("Runtime")
    axes[0].tick_params(axis="x", rotation=45)

    for method in ok_methods:
        data = benchmark_results[method]
        axes[1].plot(
            np.asarray(data["times"]) * 1e6,
            np.asarray(data["n_mean"]),
            marker="o",
            linewidth=1.6,
            label=method,
        )
    axes[1].set_xlabel("time (us)")
    axes[1].set_ylabel(r"mean $\langle n_r \rangle$")
    axes[1].set_title("Mean Rydberg occupation")
    axes[1].legend(fontsize=8)

    err_mean = np.asarray([benchmark_results[m].get("max_abs_diff_n_mean", np.nan) for m in ok_methods], dtype=float)
    axes[2].bar(ok_methods, err_mean, color="tab:orange", alpha=0.85)
    axes[2].set_yscale("symlog", linthresh=1e-12)
    axes[2].set_ylabel("max abs diff vs exact")
    axes[2].set_title("Mean occupation error")
    axes[2].tick_params(axis="x", rotation=45)
    plt.show()

    fastest = min(ok_methods, key=lambda m: benchmark_results[m]["elapsed_s"])
    comparable = [m for m in ok_methods if m != "exact" and np.isfinite(benchmark_results[m].get("max_abs_diff_n_mean", np.nan))]
    print(f"fastest successful backend: {fastest} ({benchmark_results[fastest]['elapsed_s']:.3f} s)")
    if comparable:
        best = min(comparable, key=lambda m: benchmark_results[m]["max_abs_diff_n_mean"])
        worst = max(comparable, key=lambda m: benchmark_results[m]["max_abs_diff_n_mean"])
        print(f"most consistent mean occupation: {best} ({benchmark_results[best]['max_abs_diff_n_mean']:.3e})")
        print(f"largest mean-occupation deviation: {worst} ({benchmark_results[worst]['max_abs_diff_n_mean']:.3e})")
